# 06.3 — Stochastic vs deterministic placement

The VAE's sampling ([06.2](06.2_vae_and_the_elbo.ipynb)) can sit in one of two places in the graph, and the choice defines two different architectures. **Stochastic Encoder** samples the latent and feeds the *sample* to the classifier. **Deterministic** feeds the *mean* to the classifier and only samples for the reconstruction branch. The Optimal config uses Stochastic (`EncoderOutputType='Stochastic'`), and this notebook is about why the placement matters.

This notebook covers:

* The two graph topologies, drawn.
* What each feeds the classifier — sample vs mean.
* Why Stochastic regularizes the classifier (and Deterministic doesn't).
* The `encoder_output_type` config and where the branch lives.

**Prerequisites:** [06.2 VAE + ELBO](06.2_vae_and_the_elbo.ipynb).

## Section 1 — What MATLAB does

`cfg.EncoderOutputType` selects the topology:

```matlab
switch EncoderOutputType
    case 'Stochastic'
        Z = samplingLayer(stats);       % sample
        Yclass = classifier(Z);          % classify the SAMPLE
    case 'Deterministic'
        [Z, mu] = samplingLayer(stats);
        Yclass = classifier(mu);         % classify the MEAN
end
Yrec = decoder(Z);                       % reconstruction always uses the sample
```

The reconstruction branch always uses the sample (that's what makes it a VAE). The difference is purely what the *classifier* sees — sample (Stochastic) or mean (Deterministic). Optimal uses Stochastic.

## Section 2 — The Python concepts you need

### 2.1 — The two topologies

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, (axS, axD) = plt.subplots(1, 2, figsize=(12, 4))

def node(ax, x, y, label, color):
    ax.add_patch(mpatches.FancyBboxPatch((x-0.7, y-0.28), 1.4, 0.56,
        boxstyle="round,pad=0.05", facecolor=color, edgecolor="black", lw=1.2))
    ax.text(x, y, label, ha="center", va="center", fontsize=10, fontweight="bold")

def arrow(ax, x1, y1, x2, y2, label="", color="black"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle="->", color=color, lw=1.4))
    if label: ax.text((x1+x2)/2, (y1+y2)/2+0.18, label, ha="center", fontsize=9, color=color)

for ax, title in [(axS, "Stochastic (Optimal)"), (axD, "Deterministic")]:
    node(ax, 1.5, 3.0, "encoder →\nstats", "#cce4ff")
    node(ax, 1.5, 1.5, "sample z", "#ffe4cc")
    node(ax, 4.5, 3.5, "decoder\n(reconstruction)", "#e6f0d0")
    node(ax, 4.5, 1.0, "classifier", "#f0d0e6")
    arrow(ax, 1.5, 2.7, 1.5, 1.8)                              # stats → sample
    arrow(ax, 2.2, 1.6, 3.8, 3.3, "z", "#aa0000")             # z → decoder (always)
    ax.set_title(title, fontsize=12); ax.set_xlim(0, 6); ax.set_ylim(0.4, 4.2); ax.axis("off")

# Stochastic: classifier gets z (the sample)
arrow(axS, 2.2, 1.4, 3.8, 1.1, "z (SAMPLE)", "#aa0000")
# Deterministic: classifier gets mu (the mean), drawn from the stats node
arrow(axD, 2.2, 2.9, 3.8, 1.2, "mu (MEAN)", "#3366aa")
arrow(axD, 2.2, 1.5, 3.8, 3.2, "z", "#aa0000")

plt.tight_layout(); plt.show()
print("Both: decoder gets the sample z. Difference: classifier gets z (Stochastic) or mu (Deterministic).")

### 2.2 — What the classifier sees

Stochastic feeds the classifier a *noisy* latent (the sample varies each forward, [06.2 §2.2](06.2_vae_and_the_elbo.ipynb)); Deterministic feeds it the stable mean. Demonstrate the difference in classifier input across forwards:

In [ ]:
import torch
from neural_data_decoding.models.layers.sampling import SamplingLayer

stats = torch.randn(2, 6)          # [mu | logvar], latent=3
s = SamplingLayer(); s.train()

# Stochastic: classifier would see z, which changes each forward
z1, mu1, _ = s(stats)
z2, mu2, _ = s(stats)
print("Stochastic — classifier input (z) changes across forwards?", not torch.equal(z1, z2))
print("Deterministic — classifier input (mu) changes?           ", not torch.equal(mu1, mu2))

### 2.3 — Why Stochastic regularizes

Feeding the classifier a *sample* means it sees a slightly different latent every forward — it can't memorize an exact point, so it must learn to classify a *region* of latent space robustly. That's a regularizer, much like dropout: noise on the input forces smoother decision boundaries. Deterministic (mean) removes that noise, so the classifier can overfit to precise latent points. For neural decoding, where trials are few and noisy, the Stochastic regularization helps generalization — hence Optimal's choice.

The tradeoff: Stochastic's classifier gradient is noisier (the sample varies), so training can be slightly less stable. The KL term ([06.2 §2.4](06.2_vae_and_the_elbo.ipynb)) keeps the sampling noise bounded, which mitigates that.

### 2.4 — At inference, both collapse to the mean

A crucial point ([06.13](06.13_sampling_layer_deterministic_at_inference.ipynb)): in `eval()` mode the SamplingLayer returns `z = mu`, so at *inference* Stochastic and Deterministic behave identically — both classify the mean. The stochasticity is a *training-time* regularizer only; predictions are deterministic and reproducible:

In [ ]:
s.eval()
z_eval, mu_eval, _ = s(stats)
print("eval mode: z == mu?", torch.equal(z_eval, mu_eval))
print("→ at inference, Stochastic and Deterministic give the SAME classifier input (mu).")
print("  Stochasticity only shapes the model during TRAINING.")

## Section 3 — The `neural_data_decoding` implementation

The composite's forward wires the sample through to both branches — the placement is one line of routing:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/models/composite.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "z, mu, logvar = self.sampling" in line)
for k in range(i, min(i + 10, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `z, mu, logvar = self.sampling(stats)` — the sample and the mean are both available after the sampling layer.
* The decoder receives `z` (the sample) for reconstruction; the classifier's input is the placement decision.
* `EncoderOutputType='Stochastic'` in the Optimal config means the classifier sees `z`; the SamplingLayer's eval-mode `z=mu` ([06.2 §3](06.2_vae_and_the_elbo.ipynb)) makes inference deterministic regardless.
* The config carries `encoder_output_type` in `base.yaml` — one string flips the topology, the [04.1](../04_architecture/04.1_architecture_string_dispatcher.ipynb) swap-by-config pattern applied to graph structure.

## Section 4 — Hands-on exercises

### Exercise 4.1 — noisy vs stable classifier input

Over 5 forward passes in train mode, measure the standard deviation of the classifier's input under Stochastic (z) vs Deterministic (mu). Which is zero?

In [ ]:
# Reveal:
s = SamplingLayer(); s.train()
zs = torch.stack([s(stats)[0] for _ in range(5)])
mus = torch.stack([s(stats)[1] for _ in range(5)])
print(f"Stochastic (z)  input std across forwards: {zs.std(0).mean():.4f}  (noisy — regularizes)")
print(f"Deterministic (mu) input std across forwards: {mus.std(0).mean():.4f}  (zero — stable)")

### Exercise 4.2 — inference equivalence

Show that in eval mode, the classifier input is identical whether you'd route z or mu (because z == mu).

In [ ]:
# Reveal:
s.eval()
z, mu, _ = s(stats)
print("eval: routing z vs mu to the classifier gives identical input?", torch.equal(z, mu))

## Section 5 — Common errors

### Classifier overfits despite the VAE

Possibly Deterministic placement (classifier sees the stable mean, no sampling regularization). Optimal uses Stochastic for exactly this reason (§2.3).

### Training unstable with Stochastic

The classifier gradient is noisier because its input samples vary (§2.3). Ensure the KL term is active (it bounds the sampling variance) and the KL weight isn't collapsed to zero.

### Inference gives different predictions each run

The SamplingLayer isn't in `eval()` mode, so it's still sampling ([06.13](06.13_sampling_layer_deterministic_at_inference.ipynb)). Stochastic placement + eval mode = deterministic inference; the bug is the missing `eval()`.

### Reconstruction uses the mean instead of the sample

That breaks the VAE — reconstruction MUST use the sample z (that's what the ELBO's reconstruction term assumes). Only the *classifier's* input is the placement choice; the decoder always gets z.

### Swapping to Deterministic changes nothing at eval

Expected (§2.4) — the two topologies differ only during training. If you see no difference in *training* dynamics either, confirm the config's `encoder_output_type` is actually being read.

## Section 6 — Further reading

- [06.2 VAE and the ELBO](06.2_vae_and_the_elbo.ipynb) — the sampling layer this notebook places.
- [06.13 deterministic at inference](06.13_sampling_layer_deterministic_at_inference.ipynb) — the eval-mode behavior in depth.
- [`src/neural_data_decoding/models/composite.py`](../../src/neural_data_decoding/models/composite.py) — the forward routing.

Next up: **[06.4 the EMA prior normalization deep dive](06.4_the_ema_prior_normalization_deep_dive.ipynb)** — the scale fix that makes the four-component sum meaningful.